# Two-Phase YOLO26n Pipeline Runner

This notebook runs the Sprint 4 two-phase pipeline in English, using a YOLO26n weapon detector checkpoint for both:

- Stage 2 weapon detection inside approved person crops
- the single-stage baseline comparison on the full test split

Workflow:

1. Build Phase 1 person crops
2. Train the `hold` / `no_hold` classifier
3. Run two-phase inference on a split
4. Compare two-phase vs single-stage YOLO26n


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import textwrap

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / 'scripts').exists(), 'Run this notebook from the repository root.'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

def run_command(args):
    """Run a project command and print stdout/stderr before failing."""
    args = [str(arg) for arg in args]
    print('Running:')
    print(' '.join(args))
    print()

    result = subprocess.run(
        args,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True
    )

    print('STDOUT:')
    print(result.stdout)

    print('STDERR:')
    print(result.stderr)

    print('RETURN CODE:', result.returncode)
    result.check_returncode()


def require_cuda_device(gpu_index=0):
    """Return an Ultralytics-compatible CUDA device string, or fail with a clear diagnosis."""
    import torch

    print('Python executable:', sys.executable)
    print('Torch version:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('CUDA device count:', torch.cuda.device_count())
    print("CUDA_VISIBLE_DEVICES:", os.environ.get('CUDA_VISIBLE_DEVICES'))

    if not torch.cuda.is_available() or torch.cuda.device_count() <= gpu_index:
        raise RuntimeError(textwrap.dedent(f"""
        GPU was requested, but this notebook kernel is not using a CUDA-enabled PyTorch build.

        Current Python:
            {sys.executable}

        Current torch:
            {torch.__version__}

        What to fix:
        1. Install the CUDA-enabled PyTorch build in this exact environment, or
        2. Change the Jupyter kernel to the environment where CUDA already works.

        Validation after fixing:
            import torch
            print(torch.__version__)
            print(torch.cuda.is_available())
            print(torch.cuda.get_device_name(0))

        Expected result:
            torch.cuda.is_available() == True
        """).strip())

    print('Selected GPU:', torch.cuda.get_device_name(gpu_index))
    # Ultralytics expects GPU ids as "0", "1", etc.
    return str(gpu_index)


def show_csv(path, rows=10):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    df = pd.read_csv(path)
    print(f'{path} -> {len(df)} rows')
    return df.head(rows)


def show_text_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    display(Markdown(path.read_text(encoding='utf-8')))


In [2]:
# Main parameters
FORCE_GPU = True
GPU_INDEX = 0
DEVICE = require_cuda_device(GPU_INDEX) if FORCE_GPU else 'cpu'

SPLIT = 'test'
OUTPUT_PREFIX = SPLIT
OVERWRITE_CROPS = True

# Optional overrides
PERSON_MODEL = None
HOLD_CHECKPOINT = PROJECT_ROOT / 'runs' / 'two_phase' / 'carry_classifier' / 'best.pt'

# Use your YOLO26n best.pt here.
DEFAULT_WEAPON_MODEL = PROJECT_ROOT / 'runs' / 'weapon_detector' / 'weights' / 'best.pt'
WEAPON_MODEL = DEFAULT_WEAPON_MODEL if DEFAULT_WEAPON_MODEL.exists() else Path('runs/detect/runs/yolo26n_full/weights/best.pt')

# Optional training overrides. Keep as None to use configs/two_phase.yaml defaults.
EPOCHS = None
BATCH_SIZE = None
NUM_WORKERS = None

TWO_PHASE_PREDICTIONS = PROJECT_ROOT / 'runs' / 'two_phase' / 'predictions' / f'{OUTPUT_PREFIX}_predictions.csv'
TWO_PHASE_IMAGE_SUMMARY = PROJECT_ROOT / 'runs' / 'two_phase' / 'predictions' / f'{OUTPUT_PREFIX}_image_summary.csv'
TWO_PHASE_PIPELINE_SUMMARY = PROJECT_ROOT / 'runs' / 'two_phase' / 'predictions' / f'{OUTPUT_PREFIX}_pipeline_summary.csv'
EVAL_COMPARISON = PROJECT_ROOT / 'runs' / 'two_phase' / 'evaluation' / f'{OUTPUT_PREFIX}_comparison.csv'
EVAL_SUMMARY = PROJECT_ROOT / 'runs' / 'two_phase' / 'evaluation' / f'{OUTPUT_PREFIX}_summary.md'

print('Project root:', PROJECT_ROOT)
print('Split:', SPLIT)
print('Device:', DEVICE)
print('Weapon model:', WEAPON_MODEL)


Python executable: C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\.venv\Scripts\python.exe
Torch version: 2.7.1+cu118
CUDA available: True
CUDA device count: 1
CUDA_VISIBLE_DEVICES: None
Selected GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Project root: C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection
Split: test
Device: 0
Weapon model: runs\detect\runs\yolo26n_full\weights\best.pt


In [3]:
required_paths = [
    PROJECT_ROOT / 'scripts' / 'build_two_phase_dataset.py',
    PROJECT_ROOT / 'scripts' / 'train_carry_classifier.py',
    PROJECT_ROOT / 'scripts' / 'run_two_phase_inference.py',
    PROJECT_ROOT / 'scripts' / 'evaluate_detection_pipeline.py',
    PROJECT_ROOT / 'configs' / 'two_phase.yaml',
    PROJECT_ROOT / 'data' / 'splits' / f'{SPLIT}_manifest.csv',
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n- ' + '\n- '.join(missing))

if not Path(WEAPON_MODEL).exists():
    raise FileNotFoundError(
        'WEAPON_MODEL does not exist. Update the parameter cell with the path to your YOLO26n best.pt checkpoint.'
    )

print('All required files were found.')


All required files were found.


## Step 1. Build the Phase 1 person crops

This generates the `hold` / `no_hold` person crops and metadata under `data/interim/two_phase/`.


In [4]:
cmd = [sys.executable, 'scripts/build_two_phase_dataset.py', '--device', DEVICE]

if OVERWRITE_CROPS:
    cmd.append('--overwrite')

if PERSON_MODEL is not None:
    cmd.extend(['--person-model', PERSON_MODEL])

run_command(cmd)


Running:
C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\.venv\Scripts\python.exe scripts/build_two_phase_dataset.py --device 0 --overwrite

STDOUT:
[OK] train: images=3683 hold_crops=885 no_hold_crops=3025 stage0_miss_images=48
[OK] val: images=435 hold_crops=89 no_hold_crops=407 stage0_miss_images=5
[OK] test: images=1031 hold_crops=1466 no_hold_crops=1093 stage0_miss_images=42
[OK] Saved: C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\data\interim\two_phase\metadata\two_phase_dataset_summary.csv
[OK] Saved: C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\docs\sprint4_two_phase_dataset_summary.md

STDERR:

RETURN CODE: 0


In [5]:
metadata_root = PROJECT_ROOT / 'data' / 'interim' / 'two_phase' / 'metadata'
for split_name in ['train', 'val', 'test']:
    crop_csv = metadata_root / f'{split_name}_person_crops.csv'
    miss_csv = metadata_root / f'{split_name}_stage0_misses.csv'
    print('\n===', split_name.upper(), '===')
    if crop_csv.exists():
        crop_df = pd.read_csv(crop_csv)
        print('crops:', len(crop_df), '| labels:', crop_df['label'].value_counts().to_dict() if not crop_df.empty else {})
    else:
        print('Missing:', crop_csv)
    if miss_csv.exists():
        miss_df = pd.read_csv(miss_csv)
        print('stage0 misses:', len(miss_df))
    else:
        print('Missing:', miss_csv)



=== TRAIN ===
crops: 3910 | labels: {'no_hold': 3025, 'hold': 885}
stage0 misses: 48

=== VAL ===
crops: 496 | labels: {'no_hold': 407, 'hold': 89}
stage0 misses: 5

=== TEST ===
crops: 2559 | labels: {'hold': 1466, 'no_hold': 1093}
stage0 misses: 42


## Step 2. Train the `hold` / `no_hold` classifier

This writes the best checkpoint to `runs/two_phase/carry_classifier/best.pt`.


In [6]:
cmd = [sys.executable, 'scripts/train_carry_classifier.py', '--device', DEVICE]

if EPOCHS is not None:
    cmd.extend(['--epochs', str(EPOCHS)])
if BATCH_SIZE is not None:
    cmd.extend(['--batch-size', str(BATCH_SIZE)])
if NUM_WORKERS is not None:
    cmd.extend(['--num-workers', str(NUM_WORKERS)])

run_command(cmd)


Running:
C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\.venv\Scripts\python.exe scripts/train_carry_classifier.py --device 0

STDOUT:
[Epoch 1/12] train_loss=0.9465 val_loss=0.7067 best_f1_thr=0.49 best_f1=0.5000 gate_thr=0.49 gate_recall=0.7528 gate_f1=0.5000
[Epoch 2/12] train_loss=0.7798 val_loss=0.7544 best_f1_thr=0.73 best_f1=0.5202 gate_thr=0.70 gate_recall=0.7079 gate_f1=0.5185
[Epoch 3/12] train_loss=0.7205 val_loss=0.8029 best_f1_thr=0.76 best_f1=0.5046 gate_thr=0.66 gate_recall=0.7079 gate_f1=0.4903
[Epoch 4/12] train_loss=0.6719 val_loss=0.6583 best_f1_thr=0.69 best_f1=0.5755 gate_thr=0.66 gate_recall=0.7079 gate_f1=0.5551
[Epoch 5/12] train_loss=0.6457 val_loss=0.8010 best_f1_thr=0.81 best_f1=0.5431 gate_thr=0.81 gate_recall=0.7079 gate_f1=0.5431
[Epoch 6/12] train_loss=0.6171 val_loss=0.6743 best_f1_thr=0.65 best_f1=0.5771 gate_thr=0.65 gate_recall=0.8202 gate_f1=0.5771
[Epoch 7/12] train_loss=0.5742 val_loss=0.7635 best_f1_th

In [7]:
carry_dir = PROJECT_ROOT / 'runs' / 'two_phase' / 'carry_classifier'
metrics_path = carry_dir / 'metrics.csv'
summary_path = carry_dir / 'summary.md'
checkpoint_path = carry_dir / 'best.pt'

print('Checkpoint exists:', checkpoint_path.exists(), checkpoint_path)
display(show_csv(metrics_path, rows=12))


Checkpoint exists: True C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\carry_classifier\best.pt
C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\carry_classifier\metrics.csv -> 104 rows


,kind,epoch,split,loss,train_loss,best_f1_threshold,best_f1_precision,best_f1_recall,best_f1_f1,stage1_gate_threshold,stage1_gate_precision,stage1_gate_recall,stage1_gate_f1,threshold_policy,threshold_recall_floor,threshold,threshold_role,precision,recall,f1,accuracy,tp,fp,fn,tn
0,epoch_summary,1,val,0.706741,0.946487,0.49,0.374302,0.752809,0.500000,0.49,0.374302,0.752809,0.500000,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,epoch_summary,2,val,0.754385,0.779815,0.73,0.432836,0.651685,0.520179,0.70,0.409091,0.707865,0.518519,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,epoch_summary,3,val,0.802877,0.720469,0.76,0.426357,0.617978,0.504587,0.66,0.375000,0.707865,0.490272,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,epoch_summary,4,val,0.658331,0.671866,0.69,0.495935,0.685393,0.575472,0.66,0.456522,0.707865,0.555066,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,epoch_summary,5,val,0.801022,0.645668,0.81,0.440559,0.707865,0.543103,0.81,0.440559,0.707865,0.543103,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,epoch_summary,6,val,0.674274,0.617150,0.65,0.445122,0.820225,0.577075,0.65,0.445122,0.820225,0.577075,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,epoch_summary,7,val,0.763511,0.574247,0.81,0.504065,0.696629,0.584906,0.80,0.492188,0.707865,0.580645,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,epoch_summary,8,val,0.799119,0.564371,0.66,0.440789,0.752809,0.556017,0.66,0.440789,0.752809,0.556017,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,epoch_summary,9,val,0.725954,0.557303,0.67,0.612500,0.550562,0.579882,0.37,0.412500,0.741573,0.530120,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,epoch_summary,10,val,0.683933,0.518372,0.62,0.440000,0.865169,0.583333,0.62,0.440000,0.865169,0.583333,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
show_text_file(PROJECT_ROOT / 'runs' / 'two_phase' / 'carry_classifier' / 'summary.md')


# Hold/No-Hold Classifier Training Summary

- Train samples: `3910`
- Validation samples: `496`
- Test samples: `2559`
- Positive weight: `3.4181`
- Best checkpoint epoch: `12`
- Best-F1 validation threshold: `0.93`
- Stage 1 gate threshold: `0.89`
- Threshold policy: `recall_floor`
- Recall floor for gate selection: `0.70`

## Why this threshold was chosen

The real pipeline uses a permissive `hold/no_hold` gate. The selected Stage 1 threshold is the highest validation threshold that still preserves the configured recall floor, with F1 used as a tie-breaker among eligible thresholds.

Expected tradeoff: higher candidate flow to Stage 2, fewer collapsed-recall runs, and more false positives handled later by YOLO26n.

## Validation sweep summary

- Best-F1 validation metrics: precision `0.5938`, recall `0.6404`, F1 `0.6162`
- Gate validation metrics: precision `0.5203`, recall `0.7191`, F1 `0.6038`

| kind            |   epoch | split   |   loss |   train_loss |   best_f1_threshold |   best_f1_precision |   best_f1_recall |   best_f1_f1 |   stage1_gate_threshold |   stage1_gate_precision |   stage1_gate_recall |   stage1_gate_f1 |   threshold_policy |   threshold_recall_floor |   threshold | threshold_role   |   precision |   recall |       f1 |   accuracy |   tp |   fp |   fn |   tn |
|:----------------|--------:|:--------|-------:|-------------:|--------------------:|--------------------:|-----------------:|-------------:|------------------------:|------------------------:|---------------------:|-----------------:|-------------------:|-------------------------:|------------:|:-----------------|------------:|---------:|---------:|-----------:|-----:|-----:|-----:|-----:|
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.05 |                  |    0.266467 | 1        | 0.420804 |   0.506048 |   89 |  245 |    0 |  162 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.06 |                  |    0.271341 | 1        | 0.426859 |   0.518145 |   89 |  239 |    0 |  168 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.07 |                  |    0.273846 | 1        | 0.429952 |   0.524194 |   89 |  236 |    0 |  171 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.08 |                  |    0.277259 | 1        | 0.434146 |   0.532258 |   89 |  232 |    0 |  175 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.09 |                  |    0.277259 | 1        | 0.434146 |   0.532258 |   89 |  232 |    0 |  175 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.1  |                  |    0.280757 | 1        | 0.438424 |   0.540323 |   89 |  228 |    0 |  179 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.11 |                  |    0.279365 | 0.988764 | 0.435644 |   0.540323 |   88 |  227 |    1 |  180 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.12 |                  |    0.279743 | 0.977528 | 0.435    |   0.544355 |   87 |  224 |    2 |  183 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.13 |                  |    0.285246 | 0.977528 | 0.441624 |   0.556452 |   87 |  218 |    2 |  189 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.14 |                  |    0.287129 | 0.977528 | 0.443878 |   0.560484 |   87 |  216 |    2 |  191 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.15 |                  |    0.288079 | 0.977528 | 0.445013 |   0.5625   |   87 |  215 |    2 |  192 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.16 |                  |    0.29     | 0.977528 | 0.447301 |   0.566532 |   87 |  213 |    2 |  194 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.17 |                  |    0.291946 | 0.977528 | 0.449612 |   0.570565 |   87 |  211 |    2 |  196 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.18 |                  |    0.293919 | 0.977528 | 0.451948 |   0.574597 |   87 |  209 |    2 |  198 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.19 |                  |    0.293515 | 0.966292 | 0.450262 |   0.576613 |   86 |  207 |    3 |  200 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.2  |                  |    0.293103 | 0.955056 | 0.448549 |   0.578629 |   85 |  205 |    4 |  202 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.21 |                  |    0.294118 | 0.955056 | 0.449735 |   0.580645 |   85 |  204 |    4 |  203 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.22 |                  |    0.298246 | 0.955056 | 0.454545 |   0.58871  |   85 |  200 |    4 |  207 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.23 |                  |    0.299296 | 0.955056 | 0.455764 |   0.590726 |   85 |  199 |    4 |  208 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.24 |                  |    0.300353 | 0.955056 | 0.456989 |   0.592742 |   85 |  198 |    4 |  209 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.25 |                  |    0.302491 | 0.955056 | 0.459459 |   0.596774 |   85 |  196 |    4 |  211 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.26 |                  |    0.302491 | 0.955056 | 0.459459 |   0.596774 |   85 |  196 |    4 |  211 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.27 |                  |    0.305755 | 0.955056 | 0.463215 |   0.602823 |   85 |  193 |    4 |  214 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.28 |                  |    0.306859 | 0.955056 | 0.464481 |   0.604839 |   85 |  192 |    4 |  215 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.29 |                  |    0.310219 | 0.955056 | 0.46832  |   0.610887 |   85 |  189 |    4 |  218 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.3  |                  |    0.313653 | 0.955056 | 0.472222 |   0.616935 |   85 |  186 |    4 |  221 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.31 |                  |    0.317164 | 0.955056 | 0.47619  |   0.622984 |   85 |  183 |    4 |  224 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.32 |                  |    0.320755 | 0.955056 | 0.480226 |   0.629032 |   85 |  180 |    4 |  227 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.33 |                  |    0.321839 | 0.94382  | 0.48     |   0.633065 |   84 |  177 |    5 |  230 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.34 |                  |    0.325581 | 0.94382  | 0.48415  |   0.639113 |   84 |  174 |    5 |  233 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.35 |                  |    0.328125 | 0.94382  | 0.486957 |   0.643145 |   84 |  172 |    5 |  235 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.36 |                  |    0.330709 | 0.94382  | 0.489796 |   0.647177 |   84 |  170 |    5 |  237 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.37 |                  |    0.332016 | 0.94382  | 0.491228 |   0.649194 |   84 |  169 |    5 |  238 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.38 |                  |    0.333333 | 0.94382  | 0.492669 |   0.65121  |   84 |  168 |    5 |  239 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.39 |                  |    0.338776 | 0.932584 | 0.497006 |   0.66129  |   83 |  162 |    6 |  245 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.4  |                  |    0.338776 | 0.932584 | 0.497006 |   0.66129  |   83 |  162 |    6 |  245 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.41 |                  |    0.338776 | 0.932584 | 0.497006 |   0.66129  |   83 |  162 |    6 |  245 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.42 |                  |    0.338776 | 0.932584 | 0.497006 |   0.66129  |   83 |  162 |    6 |  245 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.43 |                  |    0.341564 | 0.932584 | 0.5      |   0.665323 |   83 |  160 |    6 |  247 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.44 |                  |    0.341564 | 0.932584 | 0.5      |   0.665323 |   83 |  160 |    6 |  247 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.45 |                  |    0.343096 | 0.921348 | 0.5      |   0.669355 |   82 |  157 |    7 |  250 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.46 |                  |    0.344538 | 0.921348 | 0.501529 |   0.671371 |   82 |  156 |    7 |  251 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.47 |                  |    0.347458 | 0.921348 | 0.504615 |   0.675403 |   82 |  154 |    7 |  253 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.48 |                  |    0.353448 | 0.921348 | 0.510903 |   0.683468 |   82 |  150 |    7 |  257 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.49 |                  |    0.354978 | 0.921348 | 0.5125   |   0.685484 |   82 |  149 |    7 |  258 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.5  |                  |    0.359649 | 0.921348 | 0.51735  |   0.691532 |   82 |  146 |    7 |  261 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.51 |                  |    0.359649 | 0.921348 | 0.51735  |   0.691532 |   82 |  146 |    7 |  261 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.52 |                  |    0.361607 | 0.910112 | 0.517572 |   0.695565 |   81 |  143 |    8 |  264 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.53 |                  |    0.361991 | 0.898876 | 0.516129 |   0.697581 |   80 |  141 |    9 |  266 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.54 |                  |    0.366972 | 0.898876 | 0.521173 |   0.703629 |   80 |  138 |    9 |  269 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.55 |                  |    0.37037  | 0.898876 | 0.52459  |   0.707661 |   80 |  136 |    9 |  271 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.56 |                  |    0.373832 | 0.898876 | 0.528053 |   0.711694 |   80 |  134 |    9 |  273 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.57 |                  |    0.379147 | 0.898876 | 0.533333 |   0.717742 |   80 |  131 |    9 |  276 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.58 |                  |    0.379808 | 0.88764  | 0.531987 |   0.719758 |   79 |  129 |   10 |  278 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.59 |                  |    0.381643 | 0.88764  | 0.533784 |   0.721774 |   79 |  128 |   10 |  279 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.6  |                  |    0.381643 | 0.88764  | 0.533784 |   0.721774 |   79 |  128 |   10 |  279 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.61 |                  |    0.381643 | 0.88764  | 0.533784 |   0.721774 |   79 |  128 |   10 |  279 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.62 |                  |    0.385366 | 0.88764  | 0.537415 |   0.725806 |   79 |  126 |   10 |  281 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.63 |                  |    0.393035 | 0.88764  | 0.544828 |   0.733871 |   79 |  122 |   10 |  285 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.64 |                  |    0.388889 | 0.865169 | 0.536585 |   0.731855 |   77 |  121 |   12 |  286 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.65 |                  |    0.390863 | 0.865169 | 0.538462 |   0.733871 |   77 |  120 |   12 |  287 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.66 |                  |    0.392857 | 0.865169 | 0.540351 |   0.735887 |   77 |  119 |   12 |  288 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.67 |                  |    0.396907 | 0.865169 | 0.54417  |   0.739919 |   77 |  117 |   12 |  290 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.68 |                  |    0.403141 | 0.865169 | 0.55     |   0.745968 |   77 |  114 |   12 |  293 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.69 |                  |    0.402116 | 0.853933 | 0.546763 |   0.745968 |   76 |  113 |   13 |  294 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.7  |                  |    0.406417 | 0.853933 | 0.550725 |   0.75     |   76 |  111 |   13 |  296 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.71 |                  |    0.406417 | 0.853933 | 0.550725 |   0.75     |   76 |  111 |   13 |  296 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.72 |                  |    0.413043 | 0.853933 | 0.556777 |   0.756048 |   76 |  108 |   13 |  299 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.73 |                  |    0.415301 | 0.853933 | 0.558824 |   0.758065 |   76 |  107 |   13 |  300 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.74 |                  |    0.426136 | 0.842697 | 0.566038 |   0.768145 |   75 |  101 |   14 |  306 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.75 |                  |    0.438596 | 0.842697 | 0.576923 |   0.778226 |   75 |   96 |   14 |  311 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.76 |                  |    0.443787 | 0.842697 | 0.581395 |   0.782258 |   75 |   94 |   14 |  313 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.77 |                  |    0.451807 | 0.842697 | 0.588235 |   0.788306 |   75 |   91 |   14 |  316 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.78 |                  |    0.453988 | 0.831461 | 0.587302 |   0.790323 |   74 |   89 |   15 |  318 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.79 |                  |    0.453416 | 0.820225 | 0.584    |   0.790323 |   73 |   88 |   16 |  319 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.8  |                  |    0.448718 | 0.786517 | 0.571429 |   0.788306 |   70 |   86 |   19 |  321 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.81 |                  |    0.45098  | 0.775281 | 0.570248 |   0.790323 |   69 |   84 |   20 |  323 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.82 |                  |    0.449664 | 0.752809 | 0.563025 |   0.790323 |   67 |   82 |   22 |  325 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.83 |                  |    0.455172 | 0.741573 | 0.564103 |   0.794355 |   66 |   79 |   23 |  328 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.84 |                  |    0.455172 | 0.741573 | 0.564103 |   0.794355 |   66 |   79 |   23 |  328 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.85 |                  |    0.467626 | 0.730337 | 0.570175 |   0.802419 |   65 |   74 |   24 |  333 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.86 |                  |    0.485075 | 0.730337 | 0.58296  |   0.8125   |   65 |   69 |   24 |  338 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.87 |                  |    0.492424 | 0.730337 | 0.588235 |   0.816532 |   65 |   67 |   24 |  340 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.88 |                  |    0.507812 | 0.730337 | 0.599078 |   0.824597 |   65 |   63 |   24 |  344 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.89 | stage1_gate      |    0.520325 | 0.719101 | 0.603774 |   0.830645 |   64 |   59 |   25 |  348 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.9  |                  |    0.53913  | 0.696629 | 0.607843 |   0.83871  |   62 |   53 |   27 |  354 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.91 |                  |    0.554545 | 0.685393 | 0.613065 |   0.844758 |   61 |   49 |   28 |  358 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.92 |                  |    0.572816 | 0.662921 | 0.614583 |   0.850806 |   59 |   44 |   30 |  363 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.93 | best_f1          |    0.59375  | 0.640449 | 0.616216 |   0.856855 |   57 |   39 |   32 |  368 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.94 |                  |    0.62069  | 0.606742 | 0.613636 |   0.862903 |   54 |   33 |   35 |  374 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.95 |                  |    0.618421 | 0.52809  | 0.569697 |   0.856855 |   47 |   29 |   42 |  378 |

## Final test metrics with gate threshold

| kind       |   epoch | split   |    loss | train_loss   |   best_f1_threshold |   best_f1_precision |   best_f1_recall |   best_f1_f1 |   stage1_gate_threshold |   stage1_gate_precision |   stage1_gate_recall |   stage1_gate_f1 | threshold_policy   |   threshold_recall_floor |   threshold |   threshold_role |   precision |   recall |       f1 |   accuracy |   tp |   fp |   fn |   tn |
|:-----------|--------:|:--------|--------:|:-------------|--------------------:|--------------------:|-----------------:|-------------:|------------------------:|------------------------:|---------------------:|-----------------:|:-------------------|-------------------------:|------------:|-----------------:|------------:|---------:|---------:|-----------:|-----:|-----:|-----:|-----:|
| final_test |      12 | test    | 2.01329 |              |                0.93 |                 nan |              nan |          nan |                    0.89 |                     nan |                  nan |              nan | recall_floor       |                      0.7 |         nan |              nan |    0.721202 | 0.294679 | 0.418402 |   0.530676 |  432 |  167 | 1034 |  926 |


## Step 3. Run two-phase inference

This uses:

- the person detector from `configs/two_phase.yaml` unless you override it
- the trained `hold` / `no_hold` classifier checkpoint
- your YOLO26n `best.pt` checkpoint as the Stage 2 weapon detector


In [9]:
cmd = [
    sys.executable,
    'scripts/run_two_phase_inference.py',
    '--split', SPLIT,
    '--device', DEVICE,
    '--weapon-model', str(WEAPON_MODEL),
    '--output-prefix', OUTPUT_PREFIX,
]

if PERSON_MODEL is not None:
    cmd.extend(['--person-model', PERSON_MODEL])

if Path(HOLD_CHECKPOINT).exists():
    cmd.extend(['--hold-checkpoint', str(HOLD_CHECKPOINT)])

run_command(cmd)


Running:
C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\.venv\Scripts\python.exe scripts/run_two_phase_inference.py --split test --device 0 --weapon-model runs\detect\runs\yolo26n_full\weights\best.pt --output-prefix test --hold-checkpoint C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\carry_classifier\best.pt

STDOUT:
[OK] Saved predictions: C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\predictions\test_predictions.csv
[OK] Saved image summary: C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\predictions\test_image_summary.csv
[OK] Saved pipeline summary: C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\predictions\test_pipeline_summary.csv
[OK] Hold threshold used: 0.89 (checkpoint_stage1_gate_threshol

In [10]:
print('Predictions preview')
display(show_csv(TWO_PHASE_PREDICTIONS, rows=10))

print('\nImage summary preview')
display(show_csv(TWO_PHASE_IMAGE_SUMMARY, rows=10))

print('\nPipeline summary preview')
display(show_csv(TWO_PHASE_PIPELINE_SUMMARY, rows=10))


Predictions preview
C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\predictions\test_predictions.csv -> 313 rows


,split,image_stem,image_path,person_index,weapon_index,person_confidence,hold_probability,carry_probability,weapon_confidence,xmin,ymin,xmax,ymax
0,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,0,0.848314,0.893086,0.893086,0.601360,633.098,404.490,648.209,439.415
1,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,1,0.848314,0.893086,0.893086,0.598871,496.401,388.751,510.903,429.653
2,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,2,0.848314,0.893086,0.893086,0.348102,564.229,358.593,586.249,382.569
3,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,0,0.817063,0.971042,0.971042,0.745849,521.669,373.884,534.401,412.954
4,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,1,0.817063,0.971042,0.971042,0.341943,590.006,366.573,613.051,381.842
5,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,2,0.817063,0.971042,0.971042,0.292901,628.698,407.729,646.561,446.157
6,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,0,0.827574,0.968032,0.968032,0.686981,514.660,369.833,526.206,407.113
7,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,1,0.827574,0.968032,0.968032,0.343934,623.804,410.216,639.862,449.057
8,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,2,0.827574,0.968032,0.968032,0.253017,620.478,170.140,639.033,184.296
9,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,4,0,0.804236,0.978461,0.978461,0.653769,630.055,405.414,650.610,441.350



Image summary preview
C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\predictions\test_image_summary.csv -> 1031 rows


,split,image_stem,image_path,stage0_image_miss,persons_detected,persons_rejected_by_hold_gate,persons_filtered_out,persons_passed_stage2,final_weapon_detections
0,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,0,0,0,0,0
1,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,2,2,2,0,0
2,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,7,6,6,1,3
3,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,6,5,5,1,3
4,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,7,6,6,1,3
5,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,6,6,6,0,0
6,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,6,6,6,0,0
7,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,8,7,7,1,3
8,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,9,8,8,1,2
9,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data\raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,8,6,6,2,2



Pipeline summary preview
C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\predictions\test_pipeline_summary.csv -> 1 rows


,split,images_processed,persons_detected,persons_rejected_by_hold_gate,persons_filtered_out,persons_passed_stage2,final_weapon_detections,stage0_miss_images,hold_threshold,carry_threshold,threshold_source
0,test,1031,2785,2145,2145,640,313,42,0.89,0.89,checkpoint_stage1_gate_threshold


## Step 4. Compare the two-phase pipeline against the single-stage YOLO26n baseline

The same YOLO26n checkpoint is used as the single-stage baseline model on the full image.


In [11]:
cmd = [
    sys.executable,
    'scripts/evaluate_detection_pipeline.py',
    '--split', SPLIT,
    '--device', DEVICE,
    '--single-stage-model', str(WEAPON_MODEL),
    '--two-phase-predictions', str(TWO_PHASE_PREDICTIONS),
    '--two-phase-image-summary', str(TWO_PHASE_IMAGE_SUMMARY),
    '--output-prefix', OUTPUT_PREFIX,
]

run_command(cmd)


Running:
C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\.venv\Scripts\python.exe scripts/evaluate_detection_pipeline.py --split test --device 0 --single-stage-model runs\detect\runs\yolo26n_full\weights\best.pt --two-phase-predictions C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\predictions\test_predictions.csv --two-phase-image-summary C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\predictions\test_image_summary.csv --output-prefix test

STDOUT:
[OK] Saved comparison CSV: C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\evaluation\test_comparison.csv
[OK] Saved summary: C:\Users\mathe\OneDrive\Documents\Documents\2026.1\cvproject\ComputerVision-CCTVgundetection\runs\two_phase\evaluation\test_summary.md

STDERR:

RETURN CODE: 0


In [12]:
comparison_df = pd.read_csv(EVAL_COMPARISON)
display(comparison_df)


,pipeline,tp,fp,fn,precision,recall,f1,detections_per_image,stage0_miss_count,stage1_rejected,stage2_passed
0,single_stage,266,349,1247,0.432520,0.175810,0.250000,0.596508,NaN,NaN,NaN
1,two_phase,40,273,1473,0.127796,0.026438,0.043812,0.303589,42.0,2145.0,640.0
2,delta_two_phase_minus_single_stage,-226,-76,226,-0.304725,-0.149372,-0.206188,-0.292919,42.0,2145.0,640.0


In [13]:
show_text_file(EVAL_SUMMARY)


# Detection Pipeline Evaluation Summary

- Split: `test`
- Evaluation IoU threshold: `0.50`

## Comparison

| pipeline                           |   tp |   fp |   fn |   precision |     recall |         f1 |   detections_per_image |   stage0_miss_count |   stage1_rejected |   stage2_passed |
|:-----------------------------------|-----:|-----:|-----:|------------:|-----------:|-----------:|-----------------------:|--------------------:|------------------:|----------------:|
| single_stage                       |  266 |  349 | 1247 |    0.43252  |  0.17581   |  0.25      |               0.596508 |                     |                   |                 |
| two_phase                          |   40 |  273 | 1473 |    0.127796 |  0.0264375 |  0.0438116 |               0.303589 |                  42 |              2145 |             640 |
| delta_two_phase_minus_single_stage | -226 |  -76 |  226 |   -0.304725 | -0.149372  | -0.206188  |              -0.292919 |                  42 |              2145 |             640 |


## Output files

After a successful run, the main artifacts are:

- `data/interim/two_phase/crops/`
- `data/interim/two_phase/metadata/`
- `runs/two_phase/carry_classifier/best.pt`
- `runs/two_phase/predictions/{split}_predictions.csv`
- `runs/two_phase/predictions/{split}_image_summary.csv`
- `runs/two_phase/evaluation/{split}_comparison.csv`
